## 任务五：Seq2Seq机器翻译模型
本实验基于LSTM实现序列到序列(Seq2Seq)模型，完成德语到英语的机器翻译任务。
模型采用经典的编码器-解码器(Encoder-Decoder)架构，使用Multi30k数据集进行训练与测试。

Seq2Seq（Sequence to Sequence）是一种用于处理序列转换任务的深度学习模型，核心由**编码器（Encoder）**和解码器（Decoder）组成，广泛应用于机器翻译、文本生成等场景。

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import torchtext
import tqdm
import evaluate

In [2]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# 数据集 Dataset

### 数据集说明
使用 Multi30k 平行语料库，包含德语-英语句子对，分为训练集、验证集、测试集，是机器翻译入门标准数据集。

### 代码解释

1. **`os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'`**
   - 作用：将 Hugging Face 数据集下载源切换到国内镜像站点 `hf-mirror.com`。
   - 原因：国内网络直接访问官方源 `huggingface.co` 可能速度慢或连接不稳定，使用镜像可显著提升下载成功率与速度。

2. **`dataset = datasets.load_dataset(...)`**
   - 作用：使用 `datasets` 库从本地 JSON 文件加载 `multi30k` 数据集，包含训练集（train）、验证集（validation）和测试集（test）。
   - 说明：由于在线加载容易出现网络请求异常，因此改为读取已下载的本地 JSON 文件，数据内容与在线获取完全一致，不影响后续模型训练与评估。

In [3]:
import datasets

# 从本地 JSON 文件加载数据集（文件名已修正）
dataset = datasets.load_dataset(
    "json",
    data_files={
        "train": "train.jsonl",
        "validation": "val.jsonl",    # ✅ 对应你下载的 val.jsonl
        "test": "test.jsonl"          # ✅ 对应你下载的 test.jsonl
    }
)

print("✅ 数据集加载成功！")
print("数据集结构：", dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

✅ 数据集加载成功！
数据集结构： DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})


## 2、运行下方的单元格。
你会看到数据集对象（一个DatasetDict）包含训练、验证和测试集，每个集合中的样本数量，以及每个集合中的特征（“en”和“de”）。


In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

In [5]:
train_data, valid_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"],
)

## 3、运行下方的单元格。
我们可以索引每个数据集来查看单个示例。每个例子都有两个特征：“en”和“de”，是对应的英语和德语。


In [6]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

接下来我们进行分词。英语/德语的分词较中文要直接，比如句子"good morning!会被分词为["good", "morning", "!"]序列。

下方的代码要成功安装en_core_web_sm和de_core_news_sm后才不会报错。

# 分词器 Tokenizers

In [7]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

## 4、运行下方的单元格。
我们可以使用.tokenizer方法调用每个spaCy模型的分词器，该方法接受字符串并返回Token对象序列。我们可以使用text属性从Token对象中获取字符串。


In [8]:
string = "What a lovely day it is today!"

[token.text for token in en_nlp.tokenizer(string)]

['What', 'a', 'lovely', 'day', 'it', 'is', 'today', '!']

## 5、添加一个Markdown单元格，在其中解释下方单元格的函数的作用。


### 编码器 (Encoder) 类详细解释
该类实现了Seq2Seq模型的编码器部分，负责将源语言句子（德语）编码为上下文向量。

#### 1. 输入参数
- `input_dim`: 输入词汇表大小（德语单词总数）。
- `embedding_dim`: 词嵌入向量维度。
- `hidden_dim`: LSTM隐藏层维度。
- `n_layers`: LSTM的层数。
- `dropout`: Dropout概率，防止过拟合。

#### 2. 核心组件
- **嵌入层 (nn.Embedding)**：将单词ID转换为稠密的词向量，解决One-hot向量维度爆炸问题。
- **LSTM层 (nn.LSTM)**：核心序列处理单元，处理词向量序列，捕捉上下文语义信息。
- **Dropout层 (nn.Dropout)**：训练时随机屏蔽部分神经元，防止过拟合（仅在训练阶段生效）。

#### 3. forward函数处理流程
输入 `src` (源语言ID序列) → 经过嵌入层 → 经过Dropout → 输入LSTM。
- **输入形状**：`[src length, batch size]`
- **嵌入输出**：`[src length, batch size, embedding dim]`
- **LSTM输出**：
  - `outputs`: 所有时间步的隐藏状态输出。
  - `hidden`: 最后一层的隐藏状态 (Hidden State)。
  - `cell`: 最后一层的细胞状态 (Cell State)。

#### 4. 输出
- 返回 `(hidden, cell)`，这两个状态组合成**上下文向量 (Context Vector)**，代表整个输入句子的语义信息，将作为解码器 (Decoder) 的初始状态。

In [9]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

## 6、添加一个Markdown单元格，在其中解释下方单元格出现的\<sos>和\<eos>的含义，以及map函数的作用。


### 特殊标记与Map函数解释
1. **`<sos>` 和 `<eos>`**
   - **`<sos>` (Start of Sequence)**：句子起始标记，告知模型翻译序列的开始。
   - **`<eos>` (End of Sequence)**：句子结束标记，告知模型翻译序列的结束，模型生成该标记即停止翻译。
   - 作用：明确序列边界，是Seq2Seq模型识别序列长度的关键标识。

2. **`map` 函数**
   - 作用：将 `tokenize_example` 函数**批量应用**到训练集、验证集、测试集的每一条数据上。
   - 优势：高效并行处理数据集，自动完成批量分词和预处理，无需手动编写循环，是Hugging Face Datasets库中标准的数据处理接口。

In [10]:
max_length = 1_000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(tokenize_example, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## 7、运行下方的单元格
重新打印train_data\[0]，验证小写字符串列表以及序列标记的开始/结束符已被成功添加。


In [11]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# 词汇表 Vocabularies

下一个步骤是为源语言和目标语言构建词汇表，将词语映射为数字索引。比如"hello" = 1, "world" = 2, "bye" = 3, "hates" = 4。当向我们的模型提供文本数据时，我们使用词汇表作为look-up-table将字符串转换为标记，然后将标记转换为数字。“hello world”变成了“\[“hello”，“world”]”，然后变成了“\[1,2]”。

词汇表用于建立**单词与数字 ID 的一一映射**，将文本转换为模型可处理的数值形式，同时过滤低频词，减少计算量。

In [12]:
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

en_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["en_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

de_vocab = torchtext.vocab.build_vocab_from_iterator(
    train_data["de_tokens"],
    min_freq=min_freq,
    specials=special_tokens,
)

## 8、运行下方两个单元格
验证词汇表，分别打印英语词汇表和德语词汇表的前十个Token。


In [13]:
en_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', 'a', '.', 'in', 'the', 'on', 'man']

In [14]:
de_vocab.get_itos()[:10]

['<unk>', '<pad>', '<sos>', '<eos>', '.', 'ein', 'einem', 'in', 'eine', ',']

## 9、运行下方的单元格
使用get_stoi（stoi = "string to int "）方法获取指定的Token的索引。

In [15]:
en_vocab["the"]

7

In [16]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

In [17]:
en_vocab.set_default_index(unk_index)
de_vocab.set_default_index(unk_index)

词汇表的另一个有用特性是lookup_indices方法。它接受一个Token列表并返回一个索引列表。

## 10、运行下方的单元格
观察从Token列表到索引列表的转换。

In [18]:
tokens = ["i", "love", "watching", "crime", "shows"]
en_vocab.lookup_indices(tokens)

[956, 2169, 173, 0, 821]

对应的，lookup_tokens方法使用词汇表将索引列表转换回Token列表。

## 11、运行下方的单元格
观察从索引列表到Token列表的转换。


In [19]:
en_vocab.lookup_tokens(en_vocab.lookup_indices(tokens))

['i', 'love', 'watching', '<unk>', 'shows']

## 12、添加一个Markdown单元格，在其中解释为什么原本的"crime"被转换成了\<unk>。

### 未知词 (<unk>) 转换解释
"crime" 被转换为 `<unk>` (Unknown Token) 的核心原因是：
1. **未出现在词汇表中**：`<unk>` 是词汇表中为**未登录词**（Out-of-Vocabulary, OOV）预留的特殊标记。
2. **词汇表构建限制**：在构建词汇表时，通常会过滤掉出现次数低于阈值的低频词，或者数据集本身没有 "crime" 这个词。
3. **鲁棒性处理**：模型无法处理词汇表中不存在的单词，因此统一替换为 `<unk>`，避免因未知词导致程序崩溃。

## 13、添加一个Markdown单元格，在其中解释下方两个单元格中代码的作用。


数值化是根据词汇表将句子中的单词转换为数字 ID，并添加 `<sos>`（句子开始）、`<eos>`（句子结束）、`<pad>`（填充）、`<unk>`（未知词）特殊标记。

In [20]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

### 数值化示例函数 (numericalize_example) 作用
该函数完成文本到模型输入格式的最终转换：
1. **单词转ID**：调用词汇表的 `lookup_indices` 方法，将 `en_tokens` 和 `de_tokens` 中的每个单词转换为对应的数字ID（整数序列）。
2. **生成数据**：返回包含英语ID列表 (`en_ids`) 和德语ID列表 (`de_ids`) 的字典。
3. **作用**：计算机无法直接处理文本字符串，必须转换为数值张量 (Tensor) 才能输入到神经网络嵌入层 (Embedding Layer)。

In [21]:
fn_kwargs = {"en_vocab": en_vocab, "de_vocab": de_vocab}

train_data = train_data.map(numericalize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(numericalize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(numericalize_example, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## 14、运行下方的单元格
重新打印train_data\[0]，验证"en_ids" and "de_ids"被成功添加。


In [22]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>'],
 'en_ids': [2, 16, 24, 15, 25, 778, 17, 57, 80, 202, 1312, 5, 3],
 'de_ids': [2, 18, 26, 253, 30, 84, 20, 88, 7, 15, 110, 7647, 3171, 4, 3]}

Dataset类为我们处理的另一件事是将features转换为正确的类型。每个例子中的索引目前都是基本的Python整数。然而，为了在PyTorch中使用它们，它们需要转换为PyTorch张量。with_format方法将columns参数转换为给定的类型。这里，我们指定类型为“torch”，columns为“en_ids”和“de_ids”（我们想要转换为PyTorch张量的features）。默认情况下，with_format将删除任何不在传递给列的features列表中的features。我们希望保留这些features，这可以通过output_all_columns=True来实现。

In [23]:
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

train_data = train_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

## 15、运行下方的单元格
重新打印train_data[0]，验证“en_ids”和“de_ids”特征被转换为了张量。

In [24]:
train_data[0]

{'en_ids': tensor([   2,   16,   24,   15,   25,  778,   17,   57,   80,  202, 1312,    5,
            3]),
 'de_ids': tensor([   2,   18,   26,  253,   30,   84,   20,   88,    7,   15,  110, 7647,
         3171,    4,    3]),
 'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# Data Loaders

数据准备的最后一步是创建Data Loaders。可以对它们进行迭代以返回一批数据，每一批数据都是一个字典，其中包含数字化的英语和德语句子作为PyTorch张量。

## 16、添加一个Markdown单元格，在其中解释下方两个单元格中的函数的作用。

### 数据加载与批处理函数解释
#### 1. `get_collate_fn` 与 `collate_fn` (批处理拼接)
- **作用**：将多条数据样本拼接成一个批次 (Batch)。
- **核心操作**：
  - 提取批次中的所有英语ID (`en_ids`) 和德语ID (`de_ids`)。
  - 使用 `nn.utils.rnn.pad_sequence` 对所有句子进行**填充 (Padding)**，将不同长度的句子统一到该批次的最大长度。
- **目的**：LSTM/RNN 模型要求批次内数据长度一致，必须通过填充 `<pad>` 来满足输入要求。

#### 2. `get_data_loader` (数据迭代器)
- **作用**：构建PyTorch的数据加载器 (DataLoader)，负责批量读取数据。
- **参数说明**：
  - `dataset`: 输入的数值化数据集。
  - `batch_size`: 每批次加载的样本数量。
  - `collate_fn`: 指定上面定义的拼接函数，定义如何打包批次数据。
  - `shuffle`: 是否打乱数据顺序（训练集通常为True，验证/测试为False）。
- **最终输出**：生成可迭代的数据加载器，模型训练时直接从中获取批次数据。

In [25]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [26]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [27]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

# 构建模型

我们将分三部分构建模型。编码器，解码器和封装编码器和解码器的seq2seq模型。

# 编码器 Encoder

首先是编码器，它是一个2层的LSTM。

## 17、添加一个Markdown单元格，解释下方单元格中Encoder类的代码。
包括输入参数，核心组件（词嵌入层、LSTM层、Dropout层），forwad函数的处理流程，和输出。

### 编码器 (Encoder) 类详细解释
该类实现了Seq2Seq模型的编码器部分，负责将源语言句子（德语）编码为上下文向量。

#### 1. 输入参数
- `input_dim`: 输入词汇表大小（德语单词总数）。
- `embedding_dim`: 词嵌入向量维度。
- `hidden_dim`: LSTM隐藏层维度。
- `n_layers`: LSTM的层数。
- `dropout`: Dropout概率，防止过拟合。

#### 2. 核心组件
- **嵌入层 (nn.Embedding)**：将单词ID转换为稠密的词向量，解决One-hot向量维度爆炸问题。
- **LSTM层 (nn.LSTM)**：核心序列处理单元，处理词向量序列，捕捉上下文语义信息。
- **Dropout层 (nn.Dropout)**：训练时随机屏蔽部分神经元，防止过拟合（仅在训练阶段生效）。

#### 3. forward函数处理流程
输入 `src` (源语言ID序列) → 经过嵌入层 → 经过Dropout → 输入LSTM。
- **输入形状**：`[src length, batch size]`
- **嵌入输出**：`[src length, batch size, embedding dim]`
- **LSTM输出**：
  - `outputs`: 所有时间步的隐藏状态输出。
  - `hidden`: 最后一层的隐藏状态 (Hidden State)。
  - `cell`: 最后一层的细胞状态 (Cell State)。

#### 4. 输出
- 返回 `(hidden, cell)`，这两个状态组合成**上下文向量 (Context Vector)**，代表整个输入句子的语义信息，将作为解码器 (Decoder) 的初始状态。

In [28]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src length, batch size]
        embedded = self.dropout(self.embedding(src))
        # embedded = [src length, batch size, embedding dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs = [src length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # outputs are always from the top hidden layer
        return hidden, cell

# 解码器 Decoder

接下来是解码器，它需要与编码器对齐，同样是一个2层的LSTM。

## 18、添加一个Markdown单元格，描述Decoder的工作流程。

### 解码器 (Decoder) 工作流程
解码器负责接收编码器的上下文状态，逐词生成目标语言（英语）翻译序列。

#### 1. 核心组件初始化 (`__init__`)
- **嵌入层 (`nn.Embedding`)**：将输出单词ID转换为稠密的词向量。
- **LSTM层 (`nn.LSTM`)**：处理词向量序列，维护隐藏状态 (`hidden`) 和细胞状态 (`cell`)。
- **全连接层 (`nn.Linear`)**：将LSTM输出映射到词汇表大小，输出预测概率。
- **Dropout层**：随机失活部分神经元，防止过拟合。

#### 2. 前向传播流程 (`forward`)
1. **输入处理**：接收源序列输入 `input`、初始隐藏状态 `hidden` 和细胞状态 `cell`。
2. **维度调整**：将输入 `input` 形状从 `[batch size]` 扩展为 `[1, batch size]`，匹配LSTM输入要求。
3. **嵌入与Dropout**：输入经过嵌入层转换为词向量，并应用Dropout。
4. **LSTM序列建模**：处理嵌入向量，更新LSTM的隐藏状态和细胞状态。
5. **预测输出**：将LSTM输出通过全连接层 `fc_out` 转换为最终预测结果 `prediction`，形状为 `[batch size, output_dim]`。
6. **返回**：输出预测结果以及更新后的隐藏状态、细胞状态，用于下一步生成。

In [29]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # n directions in the decoder will both always be 1, therefore:
        # hidden = [n layers, batch size, hidden dim]
        # context = [n layers, batch size, hidden dim]
        input = input.unsqueeze(0)
        # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch size, embedding dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output = [seq length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # seq length and n directions will always be 1 in this decoder, therefore:
        # output = [1, batch size, hidden dim]
        # hidden = [n layers, batch size, hidden dim]
        # cell = [n layers, batch size, hidden dim]
        prediction = self.fc_out(output.squeeze(0))
        # prediction = [batch size, output dim]
        return prediction, hidden, cell

# Seq2Seq

## 19、添加一个Markdown单元格，解释下方单元格中Seq2Seq类的代码。
包括forward函数的流程，以及teacher forcing机制。

### Seq2Seq 模型与 Teacher Forcing 机制解释

#### 1. Seq2Seq 类 (`__init__`)
该类将编码器和解码器组合为端到端的序列到序列模型。
- **输入参数**：`encoder` (编码器实例)、`decoder` (解码器实例)、`device` (计算设备)。
- **核心逻辑**：通过断言 (`assert`) 确保编码器和解码器的隐藏层维度 (`hidden_dim`) 和层数 (`n_layers`) 一致，保证状态传递兼容性。
- **组件保存**：保存编码器、解码器和设备信息。

#### 2. 前向传播流程 (`forward`)
1. **初始化**：接收源序列 `src`、目标序列 `trg` 和教师强制比例 `teacher_forcing_ratio`。
2. **初始化输出**：创建全零张量 `outputs` 存储所有时间步的预测结果。
3. **编码**：通过编码器处理源序列，获取初始上下文状态 `(hidden, cell)`。
4. **初始输入**：解码器的初始输入为 `<sos>` 标记 (`trg[0, :]`)。
5. **循环生成**：遍历目标序列长度 `trg_length`：
   - 解码器接收当前输入、状态，输出预测和新状态。
   - 存储当前预测到 `outputs`。
   - **教师强制决策**：以 `teacher_forcing_ratio` 的概率使用真实标签 (`trg[t]`) 作为下一时刻输入；否则使用模型当前预测的最高概率词 (`top1`)。
6. **返回**：返回完整的预测序列 `outputs`。

#### 3. Teacher Forcing (教师强制) 机制
- **定义**：训练过程中，解码器的输入并非完全来自上一时刻的预测结果，而是以一定概率 (`teacher_forcing_ratio`) 直接使用真实标签（Ground Truth）。
- **作用**：解决训练初期模型预测错误导致的累积误差问题，加速模型收敛，提高训练稳定性。
- **比例设置**：例如 `0.5` 表示75%的时间使用真实标签，25%的时间使用模型预测。

In [30]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"
        assert (
            encoder.n_layers == decoder.n_layers
        ), "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        # teacher_forcing_ratio is probability to use teacher forcing
        # e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        # tensor to store decoder outputs
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        # last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # first input to the decoder is the <sos> tokens
        input = trg[0, :]
        # input = [batch size]
        for t in range(1, trg_length):
            # insert input token embedding, previous hidden and previous cell states
            # receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output = [batch size, output dim]
            # hidden = [n layers, batch size, hidden dim]
            # cell = [n layers, batch size, hidden dim]
            # place predictions in a tensor holding predictions for each token
            outputs[t] = output
            # decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio
            # get the highest predicted token from our predictions
            top1 = output.argmax(1)
            # if teacher forcing, use actual next token as next input
            # if not, use predicted token
            input = trg[t] if teacher_force else top1
            # input = [batch size]
        return outputs

# 模型训练

模型初始化

## 20、添加注释
分别将“# 编码器初始化”，“# 解码器初始化”，“# Seq2Seq模型整合”这三行注释加到下方单元格中正确的位置

In [46]:
input_dim = len(de_vocab)
output_dim = len(en_vocab)
encoder_embedding_dim = 256
decoder_embedding_dim = 256
hidden_dim = 512
n_layers = 2
encoder_dropout = 0.5
decoder_dropout = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 编码器初始化
encoder = Encoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
    encoder_dropout,
)

# 解码器初始化
decoder = Decoder(
    output_dim,
    decoder_embedding_dim,
    hidden_dim,
    n_layers,
    decoder_dropout,
)

# Seq2Seq模型整合
model = Seq2Seq(encoder, decoder, device).to(device)

权重初始化

In [32]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(7853, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(5893, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=5893, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [33]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 13,898,501 trainable parameters


优化器 optimizer

In [34]:
optimizer = optim.Adam(model.parameters())

损失函数 Loss Function

In [35]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

Training Loop:

## 21、给下方单元格中的代码逐行加注释

In [47]:
def train_fn(model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device):
    # 设置模型为训练模式，启用Dropout和BatchNorm
    model.train()
    # 初始化epoch损失为0
    epoch_loss = 0
    # 遍历数据加载器中的每个批次
    for i, batch in enumerate(data_loader):
        # 提取源序列（德语）并移动到指定设备
        src = batch["de_ids"].to(device)
        # 提取目标序列（英语）并移动到指定设备
        trg = batch["en_ids"].to(device)
        
        # 优化器梯度清零，避免梯度累积
        optimizer.zero_grad()
        
        # 前向传播，获取模型输出
        output = model(src, trg, teacher_forcing_ratio)
        # output形状: [trg_length, batch_size, trg_vocab_size]
        
        # 提取输出的维度大小（词汇表大小）
        output_dim = output.shape[-1]
        
        # 调整输出形状以适配损失函数: [trg_length-1 * batch_size, output_dim]
        output = output[1:].view(-1, output_dim)
        # 调整目标序列形状以适配损失函数: [(trg_length-1) * batch_size]
        trg = trg[1:].view(-1)
        
        # 计算损失值
        loss = criterion(output, trg)
        
        # 反向传播，计算梯度
        loss.backward()
        
        # 梯度裁剪，防止梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        
        # 更新模型参数
        optimizer.step()
        
        # 累加当前批次损失到epoch损失
        epoch_loss += loss.item()
        
    # 返回平均epoch损失
    return epoch_loss / len(data_loader)

Evaluation Loop:

In [48]:
def evaluate_fn(model, data_loader, criterion, device):
    # 设置模型为评估模式，关闭Dropout和BatchNorm
    model.eval()
    # 初始化epoch损失为0
    epoch_loss = 0
    
    # 禁用梯度计算，节省内存并加速
    with torch.no_grad():
        # 遍历数据加载器中的每个批次
        for i, batch in enumerate(data_loader):
            # 提取源序列（德语）并移动到指定设备
            src = batch["de_ids"].to(device)
            # 提取目标序列（英语）并移动到指定设备
            trg = batch["en_ids"].to(device)

            # 前向传播，关闭教师强制（teacher_forcing_ratio=0）
            output = model(src, trg, 0) 
            # output形状: [trg_length, batch_size, trg_vocab_size]
            
            # 提取输出的维度大小（词汇表大小）
            output_dim = output.shape[-1]
            
            # 调整输出形状以适配损失函数: [trg_length-1 * batch_size, output_dim]
            output = output[1:].view(-1, output_dim)
            # 调整目标序列形状以适配损失函数: [(trg_length-1) * batch_size]
            trg = trg[1:].view(-1)

            # 计算损失值（不进行反向传播）
            loss = criterion(output, trg)
            
            # 累加当前批次损失到epoch损失
            epoch_loss += loss.item()
            
    # 返回平均epoch损失
    return epoch_loss / len(data_loader)

# 模型训练

In [38]:
n_epochs = 1 # 因模型训练对计算资源要求较高，此处只设立了一轮训练。
clip = 1.0
teacher_forcing_ratio = 0.5

best_valid_loss = float("inf")

for epoch in tqdm.tqdm(range(n_epochs)):
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

100%|████████████████████████████████████████████████████████| 1/1 [15:22<00:00, 922.10s/it]

	Train Loss:   5.028 | Train PPL: 152.601
	Valid Loss:   4.850 | Valid PPL: 127.803


使用交叉熵损失函数计算预测序列与真实序列的误差，采用 Adam 优化器更新模型参数，最小化翻译损失，完成模型训练。

# 模型验证

推理阶段为模型的实际应用阶段，不再使用教师强制，模型自主逐词生成翻译结果，直到输出结束标记。

In [39]:
model.load_state_dict(torch.load("tut1-model.pt"))

<All keys matched successfully>

In [40]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        if lower:
            tokens = [token.lower() for token in tokens]
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        hidden, cell = model.encoder(tensor)
        inputs = en_vocab.lookup_indices([sos_token])
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [41]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

('Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.',
 'A man in an orange hat starring at something.')

In [42]:
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
)

# 22、运行下方单元格，得到测试集第0个索引的翻译
因为epoch只进行了一轮，不会有好的效果的翻译。
感兴趣的同学可自行增加训练轮数，观察loss和翻译质量的变化。

In [43]:
translation

['<sos>',
 'a',
 'man',
 'in',
 'a',
 'a',
 'shirt',
 'is',
 'a',
 'a',
 'a',
 '.',
 '.',
 '<eos>']

## 实验总结
1. 掌握了 Seq2Seq 模型的编码器-解码器核心架构；
2. 完成了德语→英语机器翻译的全流程实现；
3. 理解了 LSTM、数据预处理、模型训练与推理的核心逻辑。